In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

  2301952/170498071 ━━━━━━━━━━━━━━━━━━━━ 55:18 20us/step

In [ ]:
IMG_SIZE = 96

x_train_resized = tf.image.resize(
    x_train, (IMG_SIZE, IMG_SIZE)
)

x_test_resized = tf.image.resize(
    x_test, (IMG_SIZE, IMG_SIZE)
)

print(x_train_resized.shape)

(50000, 96, 96, 3)


In [ ]:
x_train_preprocessed = preprocess_input(
    x_train_resized
)

x_test_preprocessed = preprocess_input(
    x_test_resized
)

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

In [ ]:
base_model.trainable = False

print("Base model trainable:",
      base_model.trainable)

In [ ]:
inputs = layers.Input(shape=(96, 96, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    10,
    activation='softmax'
)(x)

transfer_model = models.Model(
    inputs,
    outputs
)

transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_transfer = transfer_model.fit(
    x_train_preprocessed,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
transfer_loss, transfer_accuracy = transfer_model.evaluate(
    x_test_preprocessed,
    y_test
)

print("MobileNetV2 Test Accuracy:",
      transfer_accuracy * 100, "%")

In [ ]:
scratch_loss, scratch_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Scratch CNN Accuracy:",
      scratch_accuracy * 100, "%")

print("MobileNetV2 Accuracy:",
      transfer_accuracy * 100, "%")

In [ ]:
models_names = [
    'CNN From Scratch',
    'MobileNetV2'
]

accuracies = [
    scratch_accuracy * 100,
    transfer_accuracy * 100
]

plt.figure(figsize=(8, 5))

plt.bar(models_names, accuracies)

plt.title('CNN vs MobileNetV2')
plt.ylabel('Test Accuracy (%)')
plt.ylim(0, 100)

for i, value in enumerate(accuracies):
    plt.text(i, value + 1, f'{value:.2f}%',
             ha='center')

plt.show()

In [ ]:
predictions = transfer_model.predict(
    x_test_preprocessed[:9]
)

predicted_classes = np.argmax(
    predictions,
    axis=1
)

plt.figure(figsize=(10, 10))

for i in range(9):

    plt.subplot(3, 3, i + 1)

    plt.imshow(x_test[i])

    actual = class_names[int(y_test[i][0])]
    predicted = class_names[predicted_classes[i]]

    confidence = np.max(predictions[i]) * 100

    plt.title(
        f"Actual: {actual}\n"
        f"Predicted: {predicted}\n"
        f"Confidence: {confidence:.1f}%"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()